In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import torch
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

In [2]:



# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────

DATA_DIR   = "/kaggle/input/datasets/msambare/fer2013"          # dossier racine contenant train/ et test/
BATCH_SIZE = 64
NUM_WORKERS = 4
IMG_SIZE   = 48              # taille native FER2013

# Statistiques FER2013 (niveaux de gris)
FER_MEAN = (0.507,)
FER_STD  = (0.252,)

# Les 7 classes dans l'ordre alphabétique (ImageFolder les trie ainsi)
CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


# ─────────────────────────────────────────────
#  TRANSFORMS
# ─────────────────────────────────────────────

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # force grayscale au cas où
    transforms.RandomHorizontalFlip(p=0.5),        # symétrie horizontale
    transforms.RandomRotation(degrees=10),         # légère rotation ±10°
    transforms.RandomResizedCrop(                  # zoom aléatoire léger
        size=IMG_SIZE,
        scale=(0.85, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.ColorJitter(                        # variabilité d'éclairage
        brightness=0.3,
        contrast=0.3
    ),
    transforms.ToTensor(),                         # [0,255] → [0.0, 1.0]
    transforms.Normalize(FER_MEAN, FER_STD),       # normalisation FER2013
])

# Pas d'augmentation sur val/test : on évalue le vrai modèle
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(FER_MEAN, FER_STD),
])


# ─────────────────────────────────────────────
#  DATASETS  (ImageFolder lit automatiquement
#  les sous-dossiers comme classes)
# ─────────────────────────────────────────────

def load_datasets(data_dir: str = DATA_DIR):
    train_dir = os.path.join(data_dir, "train")
    test_dir  = os.path.join(data_dir, "test")

    train_dataset = datasets.ImageFolder(
        root=train_dir,
        transform=train_transforms
    )
    test_dataset = datasets.ImageFolder(
        root=test_dir,
        transform=test_transforms
    )

    print(f"Classes détectées : {train_dataset.classes}")
    print(f"Train : {len(train_dataset)} images")
    print(f"Test  : {len(test_dataset)} images")

    return train_dataset, test_dataset


# ─────────────────────────────────────────────
#  GESTION DU DÉSÉQUILIBRE
#  WeightedRandomSampler : les classes rares
#  sont tirées plus souvent pendant l'entraînement
# ─────────────────────────────────────────────

def make_weighted_sampler(dataset: datasets.ImageFolder) -> WeightedRandomSampler:
    # Compte le nombre d'images par classe
    targets = dataset.targets                          # liste des labels (int)
    class_counts = Counter(targets)                    # {0: 3995, 1: 436, ...}

    n_total = len(targets)
    n_classes = len(class_counts)

    # Poids de classe : inversement proportionnel à la taille
    # w_c = N_total / (N_classes × N_c)
    class_weights = {
        cls: n_total / (n_classes * count)
        for cls, count in class_counts.items()
    }

    # Affichage des poids pour vérification
    print("\nPoids de classes (WeightedRandomSampler) :")
    for cls_idx, weight in sorted(class_weights.items()):
        cls_name = dataset.classes[cls_idx]
        count    = class_counts[cls_idx]
        print(f"  {cls_name:10s}  {count:5d} imgs  →  poids = {weight:.3f}")

    # Un poids par image (pas par classe)
    sample_weights = torch.tensor(
        [class_weights[t] for t in targets],
        dtype=torch.float
    )

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True          # tirage avec remise (nécessaire pour over-sampling)
    )
    return sampler


# ─────────────────────────────────────────────
#  POIDS POUR LA LOSS (CrossEntropyLoss)
#  Deuxième couche de protection contre
#  le déséquilibre, côté gradient
# ─────────────────────────────────────────────

def compute_class_weights(dataset: datasets.ImageFolder) -> torch.Tensor:
    targets = dataset.targets
    class_counts = Counter(targets)
    n_total  = len(targets)
    n_classes = len(class_counts)

    weights = torch.zeros(n_classes)
    for cls_idx, count in class_counts.items():
        weights[cls_idx] = n_total / (n_classes * count)

    # Normaliser pour que la somme = n_classes (convention courante)
    weights = weights / weights.sum() * n_classes

    print("\nPoids pour CrossEntropyLoss :")
    for i, (w, name) in enumerate(zip(weights, dataset.classes)):
        print(f"  {name:10s}  →  {w:.4f}")

    return weights


# ─────────────────────────────────────────────
#  DATALOADERS
# ─────────────────────────────────────────────

def build_dataloaders(data_dir: str = DATA_DIR):
    train_dataset, test_dataset = load_datasets(data_dir)

    sampler = make_weighted_sampler(train_dataset)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,           # remplace shuffle=True
        num_workers=NUM_WORKERS,
        pin_memory=True,           # accélère le transfert CPU→GPU
        drop_last=True             # évite les mini-batchs incomplets
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,             # toujours False pour l'évaluation
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    # Poids pour la loss (à passer au modèle)
    class_weights = compute_class_weights(train_dataset)

    return train_loader, test_loader, class_weights, train_dataset.classes


# ─────────────────────────────────────────────
#  VÉRIFICATION RAPIDE
# ─────────────────────────────────────────────

def verify_batch(loader: DataLoader, class_names: list):
    images, labels = next(iter(loader))

    print(f"\n--- Vérification du batch ---")
    print(f"Shape images : {images.shape}")    # [64, 1, 48, 48]
    print(f"Shape labels : {labels.shape}")    # [64]
    print(f"Min pixel    : {images.min():.3f}")
    print(f"Max pixel    : {images.max():.3f}")
    print(f"Mean pixel   : {images.mean():.3f}")

    batch_classes = Counter(labels.tolist())
    print(f"\nDistribution dans ce batch :")
    for idx, count in sorted(batch_classes.items()):
        print(f"  {class_names[idx]:10s}: {count}")


# ─────────────────────────────────────────────
#  POINT D'ENTRÉE
# ─────────────────────────────────────────────

if __name__ == "__main__":
    train_loader, test_loader, class_weights, class_names = build_dataloaders()
    verify_batch(train_loader, class_names)

    print(f"\nTrain batches : {len(train_loader)}")
    print(f"Test  batches : {len(test_loader)}")
    print(f"\nPrêt pour l'entraînement.")

Classes détectées : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train : 28709 images
Test  : 7178 images

Poids de classes (WeightedRandomSampler) :
  angry        3995 imgs  →  poids = 1.027
  disgust       436 imgs  →  poids = 9.407
  fear         4097 imgs  →  poids = 1.001
  happy        7215 imgs  →  poids = 0.568
  neutral      4965 imgs  →  poids = 0.826
  sad          4830 imgs  →  poids = 0.849
  surprise     3171 imgs  →  poids = 1.293

Poids pour CrossEntropyLoss :
  angry       →  0.4800
  disgust     →  4.3982
  fear        →  0.4681
  happy       →  0.2658
  neutral     →  0.3862
  sad         →  0.3970
  surprise    →  0.6047

--- Vérification du batch ---
Shape images : torch.Size([64, 1, 48, 48])
Shape labels : torch.Size([64])
Min pixel    : -2.012
Max pixel    : 1.956
Mean pixel   : 0.136

Distribution dans ce batch :
  angry     : 4
  disgust   : 10
  fear      : 4
  happy     : 13
  neutral   : 13
  sad       : 7
  surprise  : 13

Train batc

In [3]:
import torch
import torch.nn as nn

In [5]:
 
# ─────────────────────────────────────────────
#  BLOC CONVOLUTIF
#  Conv → BN → Conv → BN → ReLU → MaxPool → Dropout2d
#  Réutilisé 3 fois avec des filtres différents
# ─────────────────────────────────────────────
 
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.25):
        super().__init__()
        self.block = nn.Sequential(
            # Première convolution
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
 
            # Deuxième convolution (même résolution grâce à padding=1)
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),   # divise la résolution par 2
            nn.Dropout2d(p=dropout),                 # annule des feature maps entières
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)
 

In [6]:
 
# ─────────────────────────────────────────────
#  CUSTOM CNN
#  Entrée  : (B, 1, 48, 48)   — grayscale
#  Sortie  : (B, 7)           — logits bruts
# ─────────────────────────────────────────────
 
class CustomCNN(nn.Module):
    def __init__(self, num_classes: int = 7, dropout_fc: float = 0.5):
        super().__init__()
 
        # ── Backbone convolutif ──────────────────
        # Bloc 1 : 48×48×1  → 24×24×64
        self.block1 = ConvBlock(in_channels=1,   out_channels=64)
 
        # Bloc 2 : 24×24×64 → 12×12×128
        self.block2 = ConvBlock(in_channels=64,  out_channels=128)
 
        # Bloc 3 : 12×12×128 → 6×6×256
        self.block3 = ConvBlock(in_channels=128, out_channels=256)
 
        # ── Tête de classification ───────────────
        # 6×6×256 = 9 216 valeurs après flatten
        self.classifier = nn.Sequential(
            nn.Flatten(),
 
            nn.Linear(9216, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_fc),
 
            nn.Linear(512, num_classes),   # logits bruts → CrossEntropyLoss
        )
 
        # Initialisation des poids (améliore la convergence)
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # He initialization : adapté aux activations ReLU
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)       # → (B, 64,  24, 24)
        x = self.block2(x)       # → (B, 128, 12, 12)
        x = self.block3(x)       # → (B, 256,  6,  6)
        x = self.classifier(x)   # → (B, 7)
        return x
 

In [7]:
 
# ─────────────────────────────────────────────
#  UTILITAIRES
# ─────────────────────────────────────────────
 
def count_parameters(model: nn.Module) -> int:
    """Compte uniquement les paramètres entraînables."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
 
 
def print_model_summary(model: nn.Module, input_size=(1, 1, 48, 48)):
    """Affiche les dimensions à chaque étape du forward pass."""
    print("=" * 52)
    print(f"{'Couche':<28} {'Sortie':>12}  {'Params':>8}")
    print("=" * 52)
 
    device = next(model.parameters()).device
    x = torch.zeros(input_size).to(device)

    model.eval()  # Set to evaluation mode to handle BatchNorm with batch_size=1
    
    hooks = []
    layer_info = []
 
    def hook(module, inp, out):
        n_params = sum(p.numel() for p in module.parameters() if p.requires_grad)
        if n_params > 0 or isinstance(module, nn.Flatten):
            layer_info.append((
                module.__class__.__name__,
                tuple(out.shape[1:]),
                n_params
            ))
 
    for name, module in model.named_modules():
        if not isinstance(module, (CustomCNN, ConvBlock, nn.Sequential)):
            hooks.append(module.register_forward_hook(hook))
 
    with torch.no_grad():
        model(x)
 
    for h in hooks:
        h.remove()
 
    total = 0
    for name, shape, params in layer_info:
        shape_str = str(list(shape))
        print(f"  {name:<26} {shape_str:>12}  {params:>8,}")
        total += params
 
    print("=" * 52)
    print(f"  {'Total paramètres entraînables':<26} {'':>12}  {total:>8,}")
    print("=" * 52)
 

In [8]:
# ─────────────────────────────────────────────
#  TEST RAPIDE
# ─────────────────────────────────────────────
 
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device : {device}\n")
 
    model = CustomCNN(num_classes=7).to(device)
 
    print_model_summary(model)
 
    # Test du forward pass avec un batch factice
    dummy = torch.randn(8, 1, 48, 48).to(device)
    with torch.no_grad():
        output = model(dummy)
 
    print(f"\nTest forward pass :")
    print(f"  Entrée : {list(dummy.shape)}")
    print(f"  Sortie : {list(output.shape)}")
    print(f"  ✓ Forward pass réussi")

Device : cuda

Couche                             Sortie    Params
  Conv2d                     [64, 48, 48]       576
  BatchNorm2d                [64, 48, 48]       128
  Conv2d                     [64, 48, 48]    36,864
  BatchNorm2d                [64, 48, 48]       128
  Conv2d                     [128, 24, 24]    73,728
  BatchNorm2d                [128, 24, 24]       256
  Conv2d                     [128, 24, 24]   147,456
  BatchNorm2d                [128, 24, 24]       256
  Conv2d                     [256, 12, 12]   294,912
  BatchNorm2d                [256, 12, 12]       512
  Conv2d                     [256, 12, 12]   589,824
  BatchNorm2d                [256, 12, 12]       512
  Flatten                          [9216]         0
  Linear                            [512]  4,718,592
  BatchNorm1d                       [512]     1,024
  Linear                              [7]     3,591
  Total paramètres entraînables               5,868,359

Test forward pass :
  Entrée : [8, 

In [9]:
import os
import time
import torch
import torch.nn as nn
import numpy as np
import wandb
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
 
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [17]:
# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
 
CONFIG = {
    "data_dir"          : "/kaggle/input/datasets/msambare/fer2013",
    "num_epochs"        : 80,
    "lr"                : 1e-3,
    "weight_decay"      : 1e-4,
    "patience"          : 10,
    "scheduler_patience": 5,
    "scheduler_factor"  : 0.5,
    "save_path"         : "best_model.pth",
    "label_smoothing"   : 0.1,
    "batch_size"        : 64,
    "log_images_every"  : 5,     # log des images tous les N époques
    "n_images_to_log"   : 32,    # nombre d'images par batch loggé
}
 
CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
 
CLASS_EMOJIS = {
    'angry'   : '😠',
    'disgust' : '🤢',
    'fear'    : '😨',
    'happy'   : '😊',
    'neutral' : '😐',
    'sad'     : '😢',
    'surprise': '😲',
}
 

In [11]:

# ─────────────────────────────────────────────
#  EARLY STOPPING
# ─────────────────────────────────────────────
 
class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4):
        self.patience    = patience
        self.min_delta   = min_delta
        self.counter     = 0
        self.best_loss   = float('inf')
        self.should_stop = False
 
    def step(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop
 

In [12]:

# ─────────────────────────────────────────────
#  HELPERS WANDB
# ─────────────────────────────────────────────
 
def denormalize(tensor: torch.Tensor, mean=(0.507,), std=(0.252,)) -> torch.Tensor:
    """Inverse la normalisation pour afficher les images correctement."""
    m = torch.tensor(mean).view(1, 1, 1)
    s = torch.tensor(std).view(1, 1, 1)
    return torch.clamp(tensor * s + m, 0.0, 1.0)
 
 
def log_batch_images(images: torch.Tensor, labels: torch.Tensor,
                     preds: torch.Tensor, class_names: list,
                     epoch: int, n: int = 32):
    """
    Log une grille d'images avec caption :
    - ✓ pred  si la prédiction est correcte
    - ✗ pred | true  si elle est fausse
    """
    images_np  = denormalize(images[:n].cpu()).numpy()  # (N, 1, 48, 48)
    wandb_imgs = []
 
    for i in range(min(n, len(images_np))):
        img     = images_np[i, 0]                       # (48, 48) float [0,1]
        true_lbl = class_names[labels[i].item()]
        pred_lbl = class_names[preds[i].item()]
        correct  = (labels[i].item() == preds[i].item())
 
        caption  = f"✓ {pred_lbl}" if correct else f"✗ pred:{pred_lbl} | true:{true_lbl}"
        wandb_imgs.append(wandb.Image(img, caption=caption))
 
    wandb.log({f"images/batch_epoch_{epoch:03d}": wandb_imgs}, step=epoch)
 
 
def log_confusion_matrix(all_labels, all_preds, class_names, epoch):
    """Log la matrice de confusion normalisée en image matplotlib."""
    cm            = confusion_matrix(all_labels, all_preds)
    cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
 
    fig, ax = plt.subplots(figsize=(8, 7))
    im      = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
 
    labels_fmt = [f"{CLASS_EMOJIS.get(c, '')} {c}" for c in class_names]
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(labels_fmt, rotation=35, ha='right', fontsize=9)
    ax.set_yticklabels(labels_fmt, fontsize=9)
    ax.set_xlabel("Prédit",  fontsize=11)
    ax.set_ylabel("Vrai",    fontsize=11)
    ax.set_title(f"Matrice de confusion — époque {epoch}", fontsize=12)
 
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            color = "white" if cm_normalized[i, j] > 0.5 else "black"
            ax.text(j, i, f"{cm_normalized[i, j]:.2f}\n({cm[i, j]})",
                    ha='center', va='center', fontsize=8, color=color)
 
    plt.tight_layout()
    wandb.log({"charts/confusion_matrix": wandb.Image(fig)}, step=epoch)
    plt.close(fig)
 
 
def log_per_class_f1(f1_per_class, class_names, epoch):
    """Log un bar chart F1 par classe coloré selon le niveau."""
    colors = ['#e74c3c' if f < 0.4 else '#f39c12' if f < 0.6 else '#27ae60'
              for f in f1_per_class]
 
    fig, ax = plt.subplots(figsize=(8, 4))
    bars    = ax.bar(class_names, f1_per_class, color=colors, edgecolor='none')
    ax.set_ylim(0, 1.05)
    ax.axhline(y=float(np.mean(f1_per_class)), color='gray', linestyle='--',
               linewidth=1.2, label=f'Macro avg = {np.mean(f1_per_class):.3f}')
    ax.set_ylabel("F1-score")
    ax.set_title(f"F1 par classe — époque {epoch}")
    ax.legend(fontsize=9)
 
    for bar, val in zip(bars, f1_per_class):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
 
    plt.tight_layout()
    wandb.log({"charts/f1_per_class": wandb.Image(fig)}, step=epoch)
    plt.close(fig)
 

In [13]:
 
# ─────────────────────────────────────────────
#  UNE ÉPOQUE D'ENTRAÎNEMENT
# ─────────────────────────────────────────────
 
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss       = 0.0
    correct          = 0
    total            = 0
    first_batch_data = None
 
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
 
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
 
        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
 
        if batch_idx == 0:
            first_batch_data = (
                images.detach().cpu(),
                labels.cpu(),
                preds.detach().cpu()
            )
 
    return total_loss / total, correct / total, first_batch_data
 


In [14]:
 
# ─────────────────────────────────────────────
#  UNE ÉPOQUE DE VALIDATION
# ─────────────────────────────────────────────
 
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
 
            total_loss += loss.item() * images.size(0)
            preds       = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss     = total_loss / len(loader.dataset)
    accuracy     = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    f1_macro     = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    f1_weighted  = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_per_class = f1_score(all_labels, all_preds, average=None,       zero_division=0)
 
    return avg_loss, accuracy, f1_macro, f1_weighted, f1_per_class, all_preds, all_labels
 

In [18]:
def train(config: dict = CONFIG):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    # ── Init wandb ──────────────────────────
    run = wandb.init(
        project = "fer2013-emotion-recognition-kaggle",
        name    = "CustomCNN",
        config  = config,
        tags    = ["custom-cnn", "from-scratch", "fer2013"],
    )
    wandb.config.update({"device": str(device)})
 
    # ── Données ─────────────────────────────
    train_loader, test_loader, class_weights, class_names = build_dataloaders(config["data_dir"])
 
    # ── Modèle ──────────────────────────────
    model     = CustomCNN(num_classes=7).to(device)
    criterion = nn.CrossEntropyLoss(
        weight         = class_weights.to(device),
        label_smoothing= config["label_smoothing"]
    )
    optimizer = Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )
    scheduler = ReduceLROnPlateau(
        optimizer, mode='min',
        factor  = config["scheduler_factor"],
        patience= config["scheduler_patience"],
    )
    early_stopping = EarlyStopping(patience=config["patience"])
 
    # Suivi des gradients et des poids dans wandb
    wandb.watch(model, criterion, log="all", log_freq=100)
 
    best_val_f1  = 0.0
    best_epoch   = 0
    last_epoch   = 0
    interrupted  = False
 
    # Chemins de sauvegarde
    best_path  = config["save_path"]                          # best_model.pth
    final_path = config["save_path"].replace(".pth", "_final.pth")  # best_model_final.pth
 
    def _save_checkpoint(path: str, label: str):
        """Sauvegarde localement + upload vers wandb."""
        torch.save({
            "epoch"      : epoch,
            "model_state": model.state_dict(),
            "optimizer"  : optimizer.state_dict(),
            "scheduler"  : scheduler.state_dict(),
            "val_f1"     : f1_macro,
            "val_acc"    : val_acc,
            "class_names": class_names,
            "interrupted": interrupted,
        }, path)
        wandb.save(path)
        print(f"  [{label}] sauvegardé → {path}")
 
    print(f"\n{'Ep':>4} | {'Train Loss':>10} {'Train Acc':>10} | {'Val Loss':>10} {'Val Acc':>10} {'F1 Macro':>10} | {'LR':>8}")
    print("─" * 86)
 
    try:
        for epoch in range(1, config["num_epochs"] + 1):
            last_epoch = epoch
            t0 = time.time()
 
            train_loss, train_acc, first_batch = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )
            val_loss, val_acc, f1_macro, f1_weighted, f1_per_class, all_preds, all_labels = evaluate(
                model, test_loader, criterion, device
            )
 
            scheduler.step(val_loss)
            stop       = early_stopping.step(val_loss)
            current_lr = optimizer.param_groups[0]['lr']
            elapsed    = time.time() - t0
 
            # ── Log métriques scalaires ──────────
            wandb.log({
                "train/loss"             : train_loss,
                "train/accuracy"         : train_acc,
                "train/learning_rate"    : current_lr,
                "train/epoch_time_s"     : elapsed,
 
                "val/loss"               : val_loss,
                "val/accuracy"           : val_acc,
                "val/f1_macro"           : f1_macro,
                "val/f1_weighted"        : f1_weighted,
 
                **{f"val/f1_{name}": float(score)
                   for name, score in zip(class_names, f1_per_class)},
 
                "debug/early_stop_counter": early_stopping.counter,
            }, step=epoch)
 
            # ── Log images + graphiques tous les N époques ──
            if epoch % config["log_images_every"] == 0:
                if first_batch is not None:
                    imgs, lbls, preds_cpu = first_batch
                    log_batch_images(imgs, lbls, preds_cpu, class_names,
                                     epoch, n=config["n_images_to_log"])
 
                log_confusion_matrix(all_labels, all_preds, class_names, epoch)
                log_per_class_f1(f1_per_class, class_names, epoch)
 
            # ── Sauvegarde meilleur modèle ───────
            if f1_macro > best_val_f1:
                best_val_f1 = f1_macro
                best_epoch  = epoch
                _save_checkpoint(best_path, "best")
                marker = "  ✓ best"
            else:
                marker = ""
 
            print(
                f"{epoch:>4} | {train_loss:>10.4f} {train_acc:>9.2%} "
                f"| {val_loss:>10.4f} {val_acc:>9.2%} {f1_macro:>10.4f} "
                f"| {current_lr:>8.2e}" + marker
            )
 
            if stop:
                print(f"\nEarly stopping à l'époque {epoch}.")
                break
 
    except KeyboardInterrupt:
        interrupted = True
        print(f"\n\n⚠  Interruption manuelle à l'époque {last_epoch}.")
        print("   Sauvegarde du modèle en cours...")
 
    finally:
        # ── Toujours sauvegarder le modèle final (état actuel) ──
        # même si le run a été interrompu ou stoppé tôt
        _save_checkpoint(final_path, "final")
 
        if interrupted:
            wandb.log({"debug/interrupted_at_epoch": last_epoch})
 
        # ── Rapport final (si au moins 1 époque complète) ─────
        if best_epoch > 0 and os.path.exists(best_path):
            print(f"\n── Rapport final (meilleur checkpoint : époque {best_epoch}) ──")
 
            checkpoint = torch.load(best_path, map_location=device)
            model.load_state_dict(checkpoint["model_state"])
            _, final_acc, final_f1, _, f1_per_class, all_preds, all_labels = evaluate(
                model, test_loader, criterion, device
            )
 
            log_confusion_matrix(all_labels, all_preds, class_names, epoch=best_epoch)
            log_per_class_f1(f1_per_class, class_names, epoch=best_epoch)
 
            counts = {"angry":3995, "disgust":436, "fear":4097, "happy":7215,
                      "neutral":4965, "sad":4830, "surprise":3171}
            table  = wandb.Table(columns=["Classe", "N images train", "F1-score", "Statut"])
            for name, score in zip(class_names, f1_per_class):
                status = "🟢 Bon" if score > 0.6 else "🟡 Moyen" if score > 0.4 else "🔴 Faible"
                table.add_data(name, counts.get(name, "?"), round(float(score), 4), status)
            wandb.log({"final/f1_per_class_table": table})
 
            wandb.summary["best_epoch"]      = best_epoch
            wandb.summary["best_f1"]         = best_val_f1
            wandb.summary["final_acc"]       = final_acc
            wandb.summary["final_f1"]        = final_f1
            wandb.summary["last_epoch"]      = last_epoch
            wandb.summary["interrupted"]     = interrupted
            wandb.summary["best_model_path"] = best_path
            wandb.summary["final_model_path"]= final_path
 
            print(f"\nBest  model → {best_path}  (époque {best_epoch}, F1={best_val_f1:.4f})")
            print(f"Final model → {final_path}  (époque {last_epoch}, F1={f1_macro:.4f})")
        else:
            print("\nAucun checkpoint disponible (interruption trop tôt).")
 
        print(f"Wandb run : {run.url}")
        wandb.finish()
 
    return model, class_names
 
 

In [19]:

# ─────────────────────────────────────────────
#  POINT D'ENTRÉE
# ─────────────────────────────────────────────
 
if __name__ == "__main__":
    train()

Classes détectées : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train : 28709 images
Test  : 7178 images

Poids de classes (WeightedRandomSampler) :
  angry        3995 imgs  →  poids = 1.027
  disgust       436 imgs  →  poids = 9.407
  fear         4097 imgs  →  poids = 1.001
  happy        7215 imgs  →  poids = 0.568
  neutral      4965 imgs  →  poids = 0.826
  sad          4830 imgs  →  poids = 0.849
  surprise     3171 imgs  →  poids = 1.293

Poids pour CrossEntropyLoss :
  angry       →  0.4800
  disgust     →  4.3982
  fear        →  0.4681
  happy       →  0.2658
  neutral     →  0.3862
  sad         →  0.3970
  surprise    →  0.6047

  Ep | Train Loss  Train Acc |   Val Loss    Val Acc   F1 Macro |       LR
──────────────────────────────────────────────────────────────────────────────────────


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  [best] sauvegardé → best_model.pth
   1 |     1.4448    18.07% |     2.5300     7.90%     0.1027 | 1.00e-03  ✓ best
  [best] sauvegardé → best_model.pth
   2 |     1.1756    24.05% |     2.3101    18.46%     0.1945 | 1.00e-03  ✓ best
  [best] sauvegardé → best_model.pth
   3 |     1.0845    28.66% |     2.0492    31.10%     0.2901 | 1.00e-03  ✓ best
   4 |     1.0449    31.34% |     2.1940    27.15%     0.2803 | 1.00e-03


/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)


  [best] sauvegardé → best_model.pth
   5 |     0.9965    33.98% |     2.0431    34.05%     0.3366 | 1.00e-03  ✓ best
   6 |     0.9772    35.72% |     2.1183    31.15%     0.3036 | 1.00e-03
  [best] sauvegardé → best_model.pth
   7 |     0.9561    36.98% |     2.0135    37.39%     0.3502 | 1.00e-03  ✓ best
  [best] sauvegardé → best_model.pth
   8 |     0.9338    38.46% |     1.9432    42.45%     0.3927 | 1.00e-03  ✓ best
  [best] sauvegardé → best_model.pth
   9 |     0.9171    39.51% |     1.9610    41.86%     0.3941 | 1.00e-03  ✓ best


...

/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  [best] sauvegardé → best_model.pth
  80 |     0.6522    66.84% |     1.6387    62.68%     0.5900 | 2.50e-04  ✓ best
  [final] sauvegardé → best_model_final.pth

── Rapport final (meilleur checkpoint : époque 80) ──


/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)



Best  model → best_model.pth  (époque 80, F1=0.5900)
Final model → best_model_final.pth  (époque 80, F1=0.5900)
Wandb run : https://wandb.ai/mohamedsahbi-benkalia-i/fer2013-emotion-recognition-kaggle/runs/rivlssl2


debug/early_stop_counter,▁▁▁▂▁▁▂▆▇▁▂▃▅▁▃▆▁▁▂▆▁▁▁▁▂▁▂▅▇█▁▁▁▂▁▂▃▅▁▁
train/accuracy,▁▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████
train/epoch_time_s,▃▃▂▂▂▂▃▄█▂▂▂▃▂▂▂▂▂▂▂▁▂▂▂▂▁▁▁▁▂▁▁▁▁▁▂▁▂▁▁
train/learning_rate,██████████████████████▃▃▃▃▃▃▃▃▃▃▃▃▃▁▁▁▁▁
train/loss,█▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁▃▃▃▄▄▅▆▆▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
val/f1_angry,▁▂▄▅▅▆▅▆▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇█▇▇████▇███████
val/f1_disgust,▁▁▁▁▂▂▂▂▂▂▂▃▂▃▃▄▄▃▃▃▄▄▄▄▄▄▅▅▆▆▆▇▇▇▇▆█▇██
val/f1_fear,▂▁▃▃▄▄▄▅▅▆▆▆▅▆▆▅▆▆▇▆▆▇▇▇▇▇▇▇▇▇▇█████████
val/f1_happy,▁▄▅▅▆▇▇▆▇▇▇▇█▇▇██▇███▇█▇████████████████
+6,...


In [23]:
import os
import time
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import wandb

def train(config: dict = CONFIG, resume_path: str = None, additional_epochs: int = 80):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Init wandb ──────────────────────────
    run = wandb.init(
        project = "fer2013-emotion-recognition-kaggle",
        name    = "CustomCNN_Resumed-2" if resume_path else "CustomCNN",
        config  = config,
        tags    = ["custom-cnn", "fer2013", "resumed"] if resume_path else ["custom-cnn", "from-scratch", "fer2013"],
    )
    wandb.config.update({"device": str(device)})

    # ── Données ─────────────────────────────
    train_loader, test_loader, class_weights, class_names = build_dataloaders(config["data_dir"])

    # ── Modèle ──────────────────────────────
    model     = CustomCNN(num_classes=7).to(device)
    criterion = nn.CrossEntropyLoss(
        weight         = class_weights.to(device),
        label_smoothing= config["label_smoothing"]
    )
    optimizer = Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )
    scheduler = ReduceLROnPlateau(
        optimizer, mode='min',
        factor  = config["scheduler_factor"],
        patience= config["scheduler_patience"],
    )
    early_stopping = EarlyStopping(patience=config["patience"])

    # Suivi des gradients et des poids dans wandb
    wandb.watch(model, criterion, log="all", log_freq=100)

    best_val_f1  = 0.64586
    best_epoch   = 0
    last_epoch   = 0
    start_epoch  = 1
    interrupted  = False

    # ── LOGIQUE DE REPRISE (RESUME) ─────────
    if resume_path and os.path.exists(resume_path):
        print(f"\n[INFO] Chargement du checkpoint depuis : {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device)
        
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        
        start_epoch = checkpoint["epoch"] + 1
        best_val_f1 = checkpoint.get("val_f1", 0.0) # Évite d'écraser le best_model avec un moins bon
        
        # On met à jour le nombre total d'époques
        config["num_epochs"] = start_epoch + additional_epochs - 1
        wandb.config.update({"num_epochs": config["num_epochs"]}, allow_val_change=True)
        
        print(f"[INFO] Reprise à l'époque {start_epoch}. Entraînement prévu jusqu'à l'époque {config['num_epochs']}.\n")

    # Chemins de sauvegarde
    best_path  = config["save_path"]                          
    final_path = config["save_path"].replace(".pth", "_final.pth")  

    def _save_checkpoint(path: str, label: str):
        """Sauvegarde localement + upload vers wandb."""
        torch.save({
            "epoch"      : epoch,
            "model_state": model.state_dict(),
            "optimizer"  : optimizer.state_dict(),
            "scheduler"  : scheduler.state_dict(),
            "val_f1"     : f1_macro,
            "val_acc"    : val_acc,
            "class_names": class_names,
            "interrupted": interrupted,
        }, path)
        wandb.save(path)
        print(f"  [{label}] sauvegardé → {path}")

    print(f"\n{'Ep':>4} | {'Train Loss':>10} {'Train Acc':>10} | {'Val Loss':>10} {'Val Acc':>10} {'F1 Macro':>10} | {'LR':>8}")
    print("─" * 86)

    try:
        # Modification ici : on utilise start_epoch
        for epoch in range(start_epoch, config["num_epochs"] + 1):
            last_epoch = epoch
            t0 = time.time()

            train_loss, train_acc, first_batch = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )
            val_loss, val_acc, f1_macro, f1_weighted, f1_per_class, all_preds, all_labels = evaluate(
                model, test_loader, criterion, device
            )

            scheduler.step(val_loss)
            stop       = early_stopping.step(val_loss)
            current_lr = optimizer.param_groups[0]['lr']
            elapsed    = time.time() - t0

            # ── Log métriques scalaires ──────────
            wandb.log({
                "train/loss"             : train_loss,
                "train/accuracy"         : train_acc,
                "train/learning_rate"    : current_lr,
                "train/epoch_time_s"     : elapsed,

                "val/loss"               : val_loss,
                "val/accuracy"           : val_acc,
                "val/f1_macro"           : f1_macro,
                "val/f1_weighted"        : f1_weighted,

                **{f"val/f1_{name}": float(score)
                   for name, score in zip(class_names, f1_per_class)},

                "debug/early_stop_counter": early_stopping.counter,
            }, step=epoch)

            # ── Log images + graphiques tous les N époques ──
            if epoch % config["log_images_every"] == 0:
                if first_batch is not None:
                    imgs, lbls, preds_cpu = first_batch
                    log_batch_images(imgs, lbls, preds_cpu, class_names,
                                     epoch, n=config["n_images_to_log"])

                log_confusion_matrix(all_labels, all_preds, class_names, epoch)
                log_per_class_f1(f1_per_class, class_names, epoch)

            # ── Sauvegarde meilleur modèle ───────
            if f1_macro > best_val_f1:
                best_val_f1 = f1_macro
                best_epoch  = epoch
                _save_checkpoint(best_path, "best")
                marker = "  ✓ best"
            else:
                marker = ""

            print(
                f"{epoch:>4} | {train_loss:>10.4f} {train_acc:>9.2%} "
                f"| {val_loss:>10.4f} {val_acc:>9.2%} {f1_macro:>10.4f} "
                f"| {current_lr:>8.2e}" + marker
            )

            if stop:
                print(f"\nEarly stopping à l'époque {epoch}.")
                break

    except KeyboardInterrupt:
        interrupted = True
        print(f"\n\n⚠  Interruption manuelle à l'époque {last_epoch}.")
        print("   Sauvegarde du modèle en cours...")

    finally:
        _save_checkpoint(final_path, "final")

        if interrupted:
            wandb.log({"debug/interrupted_at_epoch": last_epoch})

        if best_epoch > 0 and os.path.exists(best_path):
            print(f"\n── Rapport final (meilleur checkpoint : époque {best_epoch}) ──")

            checkpoint = torch.load(best_path, map_location=device)
            model.load_state_dict(checkpoint["model_state"])
            _, final_acc, final_f1, _, f1_per_class, all_preds, all_labels = evaluate(
                model, test_loader, criterion, device
            )

            log_confusion_matrix(all_labels, all_preds, class_names, epoch=best_epoch)
            log_per_class_f1(f1_per_class, class_names, epoch=best_epoch)

            counts = {"angry":3995, "disgust":436, "fear":4097, "happy":7215,
                      "neutral":4965, "sad":4830, "surprise":3171}
            table  = wandb.Table(columns=["Classe", "N images train", "F1-score", "Statut"])
            for name, score in zip(class_names, f1_per_class):
                status = "🟢 Bon" if score > 0.6 else "🟡 Moyen" if score > 0.4 else "🔴 Faible"
                table.add_data(name, counts.get(name, "?"), round(float(score), 4), status)
            wandb.log({"final/f1_per_class_table": table})

            wandb.summary["best_epoch"]      = best_epoch
            wandb.summary["best_f1"]         = best_val_f1
            wandb.summary["final_acc"]       = final_acc
            wandb.summary["final_f1"]        = final_f1
            wandb.summary["last_epoch"]      = last_epoch
            wandb.summary["interrupted"]     = interrupted
            wandb.summary["best_model_path"] = best_path
            wandb.summary["final_model_path"]= final_path

            print(f"\nBest  model → {best_path}  (époque {best_epoch}, F1={best_val_f1:.4f})")
            print(f"Final model → {final_path}  (époque {last_epoch}, F1={f1_macro:.4f})")
        else:
            print("\nAucun checkpoint disponible (interruption trop tôt).")

        print(f"Wandb run : {run.url}")
        wandb.finish()

    return model, class_names

In [21]:
if __name__ == "__main__":
    # Assurez-vous que CONFIG est bien défini
    final_model_path = CONFIG["save_path"].replace(".pth", "_final.pth")
    
    # Reprendre l'entraînement pour 80 époques supplémentaires
    model, class_names = train(
        config=CONFIG, 
        resume_path=final_model_path, 
        additional_epochs=80
    )

Classes détectées : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train : 28709 images
Test  : 7178 images

Poids de classes (WeightedRandomSampler) :
  angry        3995 imgs  →  poids = 1.027
  disgust       436 imgs  →  poids = 9.407
  fear         4097 imgs  →  poids = 1.001
  happy        7215 imgs  →  poids = 0.568
  neutral      4965 imgs  →  poids = 0.826
  sad          4830 imgs  →  poids = 0.849
  surprise     3171 imgs  →  poids = 1.293

Poids pour CrossEntropyLoss :
  angry       →  0.4800
  disgust     →  4.3982
  fear        →  0.4681
  happy       →  0.2658
  neutral     →  0.3862
  sad         →  0.3970
  surprise    →  0.6047

[INFO] Chargement du checkpoint depuis : best_model_final.pth
[INFO] Reprise à l'époque 81. Entraînement prévu jusqu'à l'époque 160.


  Ep | Train Loss  Train Acc |   Val Loss    Val Acc   F1 Macro |       LR
──────────────────────────────────────────────────────────────────────────────────────


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  [best] sauvegardé → best_model.pth
  81 |     0.6551    66.88% |     1.6465    62.37%     0.5912 | 2.50e-04  ✓ best
  82 |     0.6601    67.17% |     1.6400    62.20%     0.5876 | 2.50e-04
  [best] sauvegardé → best_model.pth
  83 |     0.6575    66.66% |     1.6379    63.12%     0.5998 | 2.50e-04  ✓ best
  84 |     0.6477    67.31% |     1.6397    62.34%     0.5910 | 2.50e-04


/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)


  85 |     0.6491    67.15% |     1.6423    62.71%     0.5994 | 2.50e-04
  86 |     0.6447    67.55% |     1.6367    62.79%     0.5911 | 2.50e-04
  87 |     0.6430    67.47% |     1.6330    63.04%     0.5982 | 2.50e-04
  88 |     0.6406    67.79% |     1.6359    62.59%     0.5935 | 2.50e-04
  89 |     0.6361    68.00% |     1.6439    63.01%     0.5936 | 2.50e-04


...

/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


 160 |     0.5798    75.65% |     1.5963    66.30%     0.6419 | 3.91e-06
  [final] sauvegardé → best_model_final.pth

── Rapport final (meilleur checkpoint : époque 154) ──


/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)



Best  model → best_model.pth  (époque 154, F1=0.6459)
Final model → best_model_final.pth  (époque 160, F1=0.6419)
Wandb run : https://wandb.ai/mohamedsahbi-benkalia-i/fer2013-emotion-recognition-kaggle/runs/t15arkvq


debug/early_stop_counter,▁▁▂▄▅▁▁▁▂▁▃▁▂▃▂▇▁▃▄▁▃▁▁▂▄▆▁▂▄▆▁▃▄▅▃▅▆▃▄█
train/accuracy,▁▁▁▂▂▂▂▂▃▄▅▅▄▅▄▅▅▅▆▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████
train/epoch_time_s,▂▃▃▃▄▂▂▂▂█▄▃▃▃▂▃▂▂▂▂▃▂▃▃▃▂▂▁▅▃▃▂▂▁▂▂▂▂▁▁
train/learning_rate,████▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,██▇▇▇▆▇▇▆▇▄▅▄▄▄▄▃▂▂▂▂▃▂▃▃▃▃▂▂▂▂▁▁▁▂▂▁▂▁▁
val/accuracy,▁▃▁▂▂▂▂▁▅▄▆▅▆▅▅▅▆▅▅▅▆▇▇▇▇▆▇▇▇█▇▇▇█▇█████
val/f1_angry,▁▃▂▃▃▁▁▃▃▄▄▄▅▅▅▄▅▆▅▄▅▄▅▅▃▅▇▆▇▆▆▆▆█▇▇█▇▇▇
val/f1_disgust,▁▂▁▂▁▃▅▄▃▅▅▃▅▅▄▅▆▅▇▅▆▇▆▆▅█▆▇▆▇█▇▆▆▆█▅▇█▅
val/f1_fear,▁▁▁▂▃▂▄▅▅▆▄▄▅▄▅▅▅▅▅▆▆▆▅▆▆▇▆▇▆▇▇▇█▇▇▇██▇█
val/f1_happy,▃▂▁▂▁▂▂▂▄▄▆▅▆▅▅▆▅▅▆▆▆▆▇▅▇▇▇▇▇▆▇▆▇▇█▇█▇▇█
+6,...


In [24]:
if __name__ == "__main__":
    # Assurez-vous que CONFIG est bien défini
    final_model_path = CONFIG["save_path"].replace(".pth", "_final.pth")
    
    # Reprendre l'entraînement pour 80 époques supplémentaires
    model, class_names = train(
        config=CONFIG, 
        resume_path=final_model_path, 
        additional_epochs=80
    )

Classes détectées : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train : 28709 images
Test  : 7178 images

Poids de classes (WeightedRandomSampler) :
  angry        3995 imgs  →  poids = 1.027
  disgust       436 imgs  →  poids = 9.407
  fear         4097 imgs  →  poids = 1.001
  happy        7215 imgs  →  poids = 0.568
  neutral      4965 imgs  →  poids = 0.826
  sad          4830 imgs  →  poids = 0.849
  surprise     3171 imgs  →  poids = 1.293

Poids pour CrossEntropyLoss :
  angry       →  0.4800
  disgust     →  4.3982
  fear        →  0.4681
  happy       →  0.2658
  neutral     →  0.3862
  sad         →  0.3970
  surprise    →  0.6047

[INFO] Chargement du checkpoint depuis : best_model_final.pth
[INFO] Reprise à l'époque 161. Entraînement prévu jusqu'à l'époque 240.


  Ep | Train Loss  Train Acc |   Val Loss    Val Acc   F1 Macro |       LR
──────────────────────────────────────────────────────────────────────────────────────


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  [best] sauvegardé → best_model.pth
 161 |     0.5777    75.28% |     1.5959    66.24%     0.6468 | 3.91e-06  ✓ best
 162 |     0.5734    76.02% |     1.5940    66.45%     0.6449 | 3.91e-06
 163 |     0.5706    75.62% |     1.5964    66.05%     0.6363 | 3.91e-06
 164 |     0.5859    75.66% |     1.5948    66.33%     0.6420 | 3.91e-06


/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)


 165 |     0.5721    75.67% |     1.5950    66.17%     0.6388 | 1.95e-06
 166 |     0.5741    76.18% |     1.5969    66.16%     0.6358 | 1.95e-06
 167 |     0.5754    76.18% |     1.5891    66.54%     0.6434 | 1.95e-06
 168 |     0.5776    75.92% |     1.5939    66.29%     0.6423 | 1.95e-06
  [best] sauvegardé → best_model.pth
 169 |     0.5855    75.87% |     1.5942    66.51%     0.6471 | 1.95e-06  ✓ best


...

/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


 185 |     0.5783    76.12% |     1.5926    66.70%     0.6488 | 4.88e-07

Early stopping à l'époque 185.
  [final] sauvegardé → best_model_final.pth

── Rapport final (meilleur checkpoint : époque 175) ──


/tmp/ipykernel_55/4265273501.py:59: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/image.py:407: UserWarning: Glyph 129314 (\N{NAUSEATED FACE}) missing from font(s) DejaVu Sans.
  util.ensure_matplotlib_figure(data).savefig(buf, format=self.format)



Best  model → best_model.pth  (époque 175, F1=0.6513)
Final model → best_model_final.pth  (époque 185, F1=0.6488)
Wandb run : https://wandb.ai/mohamedsahbi-benkalia-i/fer2013-emotion-recognition-kaggle/runs/r30wza0g


debug/early_stop_counter,▁▁▂▂▃▄▁▂▂▃▄▅▅▆▁▂▂▃▄▅▅▆▇▇█
train/accuracy,▁▆▃▄▄▇▇▆▅█▅▅▇▇▆▅▇▅▄▇▃▆▄▂▇
train/epoch_time_s,▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▂▂▁▁▂█▁▁▁▂
train/learning_rate,████▄▄▄▄▄▄▄▄▂▂▂▂▂▂▂▂▁▁▁▁▁
train/loss,▄▂▂▆▂▃▃▄▆▂▃▅▁▃▄▆▃▂▃▄▆▁█▅▄
val/accuracy,▃▅▁▃▂▂▅▃▅▂▅▃▄▂█▅▆▅▄▇▂▄▅▃▇
val/f1_angry,▃▆▃▃▂▂▅▄▅▁▄▄▆▄█▂▄▄▄▅▃▂▄▂▆
val/f1_disgust,█▅▃▄▃▂▄▅▆▅▇▆▆▁▇▅▇▇▆▆▃▇▆█▇
val/f1_fear,▂▄▁▂▂▁▃▃▄▃▃▃▁▅▅▃▅▁▂█▂▃▃▂▅
val/f1_happy,▂▂▄▄▁▅▄▄▂▃▄▂▂▃█▆▄▆▄█▁▅▄▅▇
+6,...
